In [2]:
import torch
import os
import numpy as np
import random
import numpy as np

In [10]:
path = '../lab 8/names'
files = [file for file in os.listdir(path)]
vocabulary = set()
for file in files:
    with open(os.path.join(path, file), 'r', encoding= 'utf-8') as f:
        for word in f.readlines():
            for char in list(word.strip()):
                vocabulary.add(char)
        
vocabulary = list(vocabulary)
input_dim = len(vocabulary) + 1

In [11]:
class Data(torch.utils.data.Dataset):
    def __init__(self, path, files, vocabulary, input_dim):
        super().__init__()
        self.files = files
        self.vocabulary, self.input_dim = vocabulary, input_dim
        self.word_tensor, self.word_data, self.label_index, self.label = [], [], [], []

        for i in range(len(self.files)):
            with open(os.path.join(path, self.files[i]), 'r', encoding= 'utf-8') as f:
                for word in f.readlines():
                    self.word_tensor.append(self.onehotencoding(word.strip()))
                    self.label_index.append(torch.tensor(i))
                    self.word_data.append(word.strip())
                    self.label.append(self.files[i])

    def onehotencoding(self, word):
        ret = torch.zeros((len(word), 1, self.input_dim))
        for i in range(len(word)):
            if word[i] in self.vocabulary:
                ret[i, 0, self.vocabulary.index(word[i])] = 1
            else:
                ret[i, 0, -1] = 1
        return ret.reshape((1, len(word), -1))
    
    def __len__(self):
        return len(self.label)
    
    def __getitem__(self, index):
        return self.word_tensor[index], self.word_data[index], self.label_index[index], self.label[index]
    

In [14]:
alldata = Data('../lab 8/names/', files, vocabulary, input_dim)
train, test = torch.utils.data.random_split(alldata, [0.9, 0.1])

In [15]:
class Network(torch.nn.Module):
    def __init__(self, input_dim, out_features):
        super().__init__()
        self.rnn = torch.nn.LSTM(input_size= input_dim, hidden_size= 128, num_layers= 1, batch_first= True)
        self.fc1 = torch.nn.Linear(in_features= 128, out_features= out_features)
    def forward(self, x):
        y, h = self.rnn(x)
        y = y[:, -1, :]
        return self.fc1(y)

In [ ]:
model = Network(input_dim= input_dim, out_features= len(files))
model = model.to('cuda')

In [17]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr= 0.001, weight_decay= 0.0001)

In [18]:
model.train()
for epoch in range(30):
    batches = list(range(len(train)))
    random.shuffle(batches)
    batches = np.array_split(batches, len(batches) // 64)
    running_loss = 0.0
    for batch in batches:
        for i in batch:
            optimizer.zero_grad()
            word_tensor, word_data, label_index, label = train[i]
            word_tensor, label_index = word_tensor.to('cuda'), label_index.unsqueeze(0).to('cuda')
            output = model(word_tensor)
            loss = criterion(output, label_index)
            loss.backward()
            running_loss += loss.item()
            optimizer.step()
    print(f'epoch - {epoch}, loss = {running_loss}')
            

epoch - 0, loss = 19308.795312615068
epoch - 1, loss = 13447.677892174404
epoch - 2, loss = 11373.050334769796
epoch - 3, loss = 10122.265168174712
epoch - 4, loss = 9343.056272547055
epoch - 5, loss = 8632.503906081498
epoch - 6, loss = 8103.784394161369
epoch - 7, loss = 7648.9165037613675
epoch - 8, loss = 7217.7614810265895
epoch - 9, loss = 6908.457649036865
epoch - 10, loss = 6644.800514306908
epoch - 11, loss = 6302.279464343864
epoch - 12, loss = 6066.073084058002
epoch - 13, loss = 5904.466971550266
epoch - 14, loss = 5722.366403918297
epoch - 15, loss = 5575.467906977858
epoch - 16, loss = 5395.13068524707
epoch - 17, loss = 5289.650575150959
epoch - 18, loss = 5193.07707953208
epoch - 19, loss = 5039.437934564921
epoch - 20, loss = 5011.979815827587
epoch - 21, loss = 4849.569681711864
epoch - 22, loss = 4810.546368649436
epoch - 23, loss = 4683.085100599162
epoch - 24, loss = 4704.264597787719
epoch - 25, loss = 4696.670555395103
epoch - 26, loss = 4567.4977619777055
epoch 

In [19]:
model.eval()
all_pred, all_label = [], []
with torch.no_grad():
    batches = list(range(len(test)))
    random.shuffle(batches)
    batches = np.array_split(batches, len(batches) // 64)
    for batch in batches:
        for i in batch:
            word_tensor, word_data, label_index, label = test[i]
            word_tensor = word_tensor.to('cuda')
            output = model(word_tensor)
            val, index = torch.max(output, dim = 1)
            all_pred.append(index.to('cpu').numpy())
            all_label.append(label_index)


In [20]:
from sklearn.metrics import accuracy_score
print(accuracy_score(all_pred, all_label))

0.8166417538614849
